# CNN-GNN-HMER CNN-GNN - Kaggle 2xT4 + DagsHub MLflow Cloud

Notebook này được dựng lại dựa trên `template.ipynb` gốc để tích hợp DagsHub MLflow (bắn log và checkpoint trực tiếp lên đám mây DagsHub):

- Giữ flow Miniconda → env `tamer` Python 3.7 → clone repo → `%cd` vào đúng project con → install → unzip CROHME → train.
- Tích hợp **DagsHub MLflow Cloud** (không dùng MLflow) để tự động đồng bộ model checkpoints và file metrics.csv lên đám mây DagsHub một cách tin cậy.
- Train mặc định bằng 2 GPU T4: `--trainer.gpus=2`.

## 0. Tham số chính

Nếu Kaggle không bật 2 GPU, sửa `GPUS = 1`. Còn mặc định cho 2xT4 là `GPUS = 2`.

In [3]:
# Tham số chính cho notebook
GPUS = 2
CONFIG = "config/crohme.yaml"
WORKDIR = "/kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/1-cnn-gnn"

# Cấu hình DagsHub MLflow Tracking
DAGSHUB_USERNAME = "KhaiHASO" # Tên đăng nhập DagsHub của bạn (mặc định trùng với GitHub)
DAGSHUB_REPO = "CNN-GNN-HMER" # Tên repository trên DagsHub của bạn
MLFLOW_TRACKING_URI = f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow"

# Token truy cập DagsHub (Lấy trong phần Settings -> Access Tokens trên DagsHub)
# Khuyên dùng: Thêm vào Kaggle Secrets dưới tên DAGSHUB_TOKEN để bảo mật hơn
DAGSHUB_TOKEN = "YOUR_DAGSHUB_TOKEN"

print("WORKDIR:", WORKDIR)
print("CONFIG:", CONFIG)
print("GPUS:", GPUS)
print("MLFLOW_TRACKING_URI:", MLFLOW_TRACKING_URI)

WORKDIR: /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/1-cnn-gnn
CONFIG: config/crohme.yaml
GPUS: 2
MLFLOW_TRACKING_URI: https://dagshub.com/KhaiHASO/CNN-GNN-HMER.mlflow


## 1. Cài đặt Miniconda

In [4]:
# Cài đặt Miniconda
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -b -f -p /kaggle/working/miniconda
!rm Miniconda3-latest-Linux-x86_64.sh

# Thêm conda vào PATH
import os
os.environ['PATH'] = "/kaggle/working/miniconda/bin:" + os.environ['PATH']

--2026-06-14 11:24:05--  https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.32.241, 104.16.191.158, 2606:4700::6810:bf9e, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.32.241|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 163179296 (156M) [application/octet-stream]
Saving to: ‘Miniconda3-latest-Linux-x86_64.sh’

Miniconda3-latest-L 100%[===================>] 155.62M   301MB/s    in 0.5s    

2026-06-14 11:24:05 (301 MB/s) - ‘Miniconda3-latest-Linux-x86_64.sh’ saved [163179296/163179296]

PREFIX=/kaggle/working/miniconda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best r

## 2. Tạo môi trường Python 3.7

In [5]:
# Accept Anaconda Terms of Service cho 2 channel mặc định
!/kaggle/working/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!/kaggle/working/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


In [6]:
# Tạo môi trường Python 3.7
!/kaggle/working/miniconda/bin/conda create -n tamer python=3.7 -y

Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - defaults
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 26.3.2
    latest version: 26.5.2

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /kaggle/working/miniconda/envs/tamer

  added / updated specs:
    - python=3.7


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-5.1          |           52_gnu           7 KB
    ca-certificates-2026.5.14  |       h06a4308_0         107 KB
    certifi-2022.12.7          |   py37h06a4308_0         150 KB
    libgcc-15.2.0              |       h69a1729_8         803 KB
    libgcc-ng-15.2.0           |       h166f726_8          28 KB
    libstdcxx-15.2.0           |       h39759b7_8         3.7 MB
    

## 3. Kiểm tra phiên bản Python và pip

In [7]:
# Kiểm tra phiên bản Python và pip
shell_script = """
source /kaggle/working/miniconda/bin/activate tamer
python --version
pip --version
"""
with open("activate_env.sh", "w") as f:
    f.write(shell_script)
!bash activate_env.sh

Python 3.7.16
pip 22.3.1 from /kaggle/working/miniconda/envs/tamer/lib/python3.7/site-packages/pip (python 3.7)


## 4. Clone repo CNN-GNN-HMER

In [31]:
%cd /kaggle/working

!rm -rf /kaggle/working/CNN-GNN-HMER
!git clone https://github.com/KhaiHASO/CNN-GNN-HMER.git /kaggle/working/CNN-GNN-HMER

/kaggle/working
Cloning into '/kaggle/working/CNN-GNN-HMER'...
remote: Enumerating objects: 1051, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 1051 (delta 54), reused 58 (delta 23), pack-reused 960 (from 1)
Receiving objects: 100% (1051/1051), 198.10 MiB | 28.62 MiB/s, done.
Resolving deltas: 100% (295/295), done.


## 5. Di chuyển vào thư mục CNN-GNN

In [32]:
# Di chuyển vào thư mục dự án con
%cd /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/1-cnn-gnn
!pwd
!ls -lh

/kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/1-cnn-gnn
/kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/1-cnn-gnn
total 40K
drwxr-xr-x 2 root root 4.0K Jun 14 11:36 config
drwxr-xr-x 2 root root 4.0K Jun 14 11:36 eval
drwxr-xr-x 2 root root 4.0K Jun 14 11:36 images
-rw-r--r-- 1 root root 2.8K Jun 14 11:36 README.md
-rw-r--r-- 1 root root 6.6K Jun 14 11:36 README-TONGHOP.md
-rw-r--r-- 1 root root  303 Jun 14 11:36 requirements.txt
-rw-r--r-- 1 root root  512 Jun 14 11:36 setup.py
drwxr-xr-x 5 root root 4.0K Jun 14 11:36 tamer
-rw-r--r-- 1 root root 3.2K Jun 14 11:36 train.py


## 6. Cài đặt các gói từ conda

In [10]:
# Cài đặt các gói từ conda
!source /kaggle/working/miniconda/bin/activate tamer && conda install pytorch-lightning=1.4.9 torchmetrics=0.6.0 -c conda-forge -y
!source /kaggle/working/miniconda/bin/activate tamer && conda install pandoc=1.19.2.1 -c conda-forge -y

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /kaggle/working/miniconda/envs/tamer

  added / updated specs:
    - pytorch-lightning=1.4.9
    - torchmetrics=0.6.0


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    absl-py-2.1.0              |     pyhd8ed1ab_0         105 KB  conda-forge
    aiohttp-3.7.4.post0        |   py37h5e8e339_1         626 KB  conda-forge
    async-timeout-3.0.1        |          py_1000          11 KB  conda-forge
    attrs-24.2.0               |     pyh71513ae_0          55 KB  conda-forge
    blinker-1.6.3              |     pyhd8ed1ab_0          18 KB  conda-forge
    brotli-python-1.0.9        |   py37hd23a5d3_7         352 KB  conda-forge
    c-ares-1.34.6              |       hb03c661_0         203 KB  conda-forge
    ca-

## 7. Fix lỗi GLIBCXX

In [11]:
# Cài đặt libstdcxx-ng để fix lỗi GLIBCXX
!source /kaggle/working/miniconda/bin/activate tamer && conda install -c conda-forge libstdcxx-ng -y

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: done

# All requested packages already installed.



## 8. Cài requirements, setup.py và W&B

In [12]:
# Cài đặt các gói từ requirements.txt và setup.py
!source /kaggle/working/miniconda/bin/activate tamer && pip install -r requirements.txt && pip install -e .

# Đảm bảo cài đặt thư viện mlflow và dagshub
!source /kaggle/working/miniconda/bin/activate tamer && pip install mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 2.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 11.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 14.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 MB 20.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 26.7 MB/s eta 0:00:0000:010:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.4/97.4 kB 12.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 73.7 MB/s eta 0:00:00ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 40.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 48.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 9. Ghi đè config/crohme.yaml chuẩn cho notebook này

In [13]:
# Backup config cũ trước khi ghi đè
!mkdir -p config_backup
!cp config/crohme.yaml config_backup/crohme.original.yaml || true

In [14]:
%%writefile config/crohme.yaml
seed_everything: 7
trainer:
  checkpoint_callback: true
  logger:
    class_path: pytorch_lightning.loggers.MLFlowLogger
    init_args:
      experiment_name: cnn-gnn-hmer-cnn-gnn
  callbacks:
    - class_path: pytorch_lightning.callbacks.LearningRateMonitor
      init_args:
        logging_interval: epoch
    - class_path: pytorch_lightning.callbacks.ModelCheckpoint
      init_args:
        save_top_k: 1
        monitor: val_ExpRate
        mode: max
        filename: 'best_model'
  gpus: 2
  accelerator: auto
  check_val_every_n_epoch: 2
  max_epochs: 100
  deterministic: true
  precision: 16
model:
  d_model: 256
  # encoder
  growth_rate: 24
  num_layers: 16
  # decoder
  nhead: 8
  num_decoder_layers: 3
  dim_feedforward: 1024
  dc: 32
  dropout: 0.3
  vocab_size: 113  # 110 + 3
  cross_coverage: true
  self_coverage: true
  # GAT (Graph Attention Network) - optional
  use_gat: true  # CNN-GNN variant: enable GAT layers
  gat_num_layers: 2
  gat_num_heads: 8
  gat_hidden_dim: null  # null means use d_model
  gat_dropout: 0.1
  # beam search
  beam_size: 10
  max_len: 150
  alpha: 1.0
  early_stopping: false
  temperature: 1.0
  # training
  learning_rate: 1.0
  patience: 20
  milestones:
    - 300
    - 350
data:
  folder: data/crohme
  test_folder: 2014
  max_size: 320000
  scale_to_limit: true
  train_batch_size: 8
  eval_batch_size: 2
  num_workers: 5
  scale_aug: false

Overwriting config/crohme.yaml


In [15]:
# Kiểm tra lại config đang dùng
!echo "===== config/crohme.yaml ====="
!cat config/crohme.yaml

===== config/crohme.yaml =====
seed_everything: 7
trainer:
  checkpoint_callback: true
  logger:
    class_path: pytorch_lightning.loggers.MLFlowLogger
    init_args:
      experiment_name: cnn-gnn-hmer-cnn-gnn
  callbacks:
    - class_path: pytorch_lightning.callbacks.LearningRateMonitor
      init_args:
        logging_interval: epoch
    - class_path: pytorch_lightning.callbacks.ModelCheckpoint
      init_args:
        save_top_k: 1
        monitor: val_ExpRate
        mode: max
        filename: '{epoch}-{step}-{val_ExpRate:.4f}'
  gpus: 2
  accelerator: auto
  check_val_every_n_epoch: 2
  max_epochs: 100
  deterministic: true
  precision: 16
model:
  d_model: 256
  # encoder
  growth_rate: 24
  num_layers: 16
  # decoder
  nhead: 8
  num_decoder_layers: 3
  dim_feedforward: 1024
  dc: 32
  dropout: 0.3
  vocab_size: 113  # 110 + 3
  cross_coverage: true
  self_coverage: true
  # GAT (Graph Attention Network) - optional
  use_gat: true  # CNN-GNN variant: enable GAT layers
  gat_nu

## 10. Cấu hình W&B riêng cho notebook này

In [23]:
# Tham số chính cho notebook
GPUS = 2
CONFIG = "config/crohme.yaml"
WORKDIR = "/kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/1-cnn-gnn"

# Cấu hình DagsHub MLflow Tracking
DAGSHUB_USERNAME = "KhaiHASO" # Tên đăng nhập DagsHub của bạn (mặc định trùng với GitHub)
DAGSHUB_REPO = "CNN-GNN-HMER" # Tên repository trên DagsHub của bạn
MLFLOW_TRACKING_URI = f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow"

# Token truy cập DagsHub (Lấy trong phần Settings -> Access Tokens trên DagsHub)
# Khuyên dùng: Thêm vào Kaggle Secrets dưới tên DAGSHUB_TOKEN để bảo mật hơn
DAGSHUB_TOKEN = "YOUR_DAGSHUB_TOKEN"

print("WORKDIR:", WORKDIR)
print("CONFIG:", CONFIG)
print("GPUS:", GPUS)
print("MLFLOW_TRACKING_URI:", MLFLOW_TRACKING_URI)

WORKDIR: /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/1-cnn-gnn
CONFIG: config/crohme.yaml
GPUS: 2
MLFLOW_TRACKING_URI: https://dagshub.com/KhaiHASO/CNN-GNN-HMER.mlflow


In [26]:
    # Cấu hình các biến môi trường cho DagsHub MLflow                                                                                          
    import os
  
    # Ưu tiên lấy token bảo mật từ Kaggle Secrets (Kaggle Add-ons -> Secrets)
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        token = user_secrets.get_secret("DAGSHUB_TOKEN")
    except Exception:
        token = DAGSHUB_TOKEN
  
    os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
    os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
    os.environ["MLFLOW_TRACKING_PASSWORD"] = token
  
    print("MLflow Tracking environment variables set!")

MLflow Tracking environment variables set!


## 11. Chuẩn bị dữ liệu CROHME

Config đọc dữ liệu ở `data/crohme`, nên cell này giải nén `CROHME.zip` từ repo root vào thư mục model hiện tại.

In [34]:
# Cách 1: Giải nén CROHME.zip có sẵn trong repo hiện tại
!mkdir -p data
!if [ -f "/kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/data/CROHME.zip" ]; then     unzip -o /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/data/CROHME.zip -d data/;   else     echo "Không thấy /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/data/CROHME.zip";     echo "Nếu dùng Kaggle Dataset, chạy cell Cách 2 bên dưới.";   fi

!echo "===== data tree ====="
!find data -maxdepth 3 -type d | sort | head -50

Archive:  /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/data/CROHME.zip
   creating: data/crohme/
   creating: data/crohme/train/
  inflating: data/crohme/train/caption.txt  
  inflating: data/crohme/train/images.pkl  
  inflating: data/crohme/dictionary.txt  
   creating: data/crohme/2014/
  inflating: data/crohme/2014/caption.txt  
  inflating: data/crohme/2014/images.pkl  
   creating: data/crohme/2019/
  inflating: data/crohme/2019/caption.txt  
  inflating: data/crohme/2019/images.pkl  
   creating: data/crohme/2016/
  inflating: data/crohme/2016/caption.txt  
  inflating: data/crohme/2016/images.pkl  
===== data tree =====
data
data/crohme
data/crohme/2014
data/crohme/2016
data/crohme/2019
data/crohme/train


In [ ]:
# Cách 2: Sử dụng từ Kaggle dataset nếu anh upload CROHME.zip riêng
# Sửa path /kaggle/input/crohme-dataset/CROHME.zip nếu dataset của anh có tên khác.
# !mkdir -p data
# !cp /kaggle/input/crohme-dataset/CROHME.zip data/
# !unzip -o data/CROHME.zip -d data/
# !find data -maxdepth 3 -type d | sort | head -50

## 12. Check nhanh repo, data, config, GPU trước khi train

In [28]:
!echo "===== PWD ====="
!pwd
!echo "===== Repo files ====="
!ls -lh
!echo "===== Config files ====="
!ls -lh config
!echo "===== Data files ====="
!find data -maxdepth 3 | head -80
!echo "===== Eval files ====="
!ls -lh eval || true

===== PWD =====
/kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/1-cnn-gnn
===== Repo files =====
total 44K
drwxr-xr-x 2 root root 4.0K Jun 14 11:32 config
drwxr-xr-x 3 root root 4.0K Jun 14 11:34 data
drwxr-xr-x 2 root root 4.0K Jun 14 11:32 eval
drwxr-xr-x 2 root root 4.0K Jun 14 11:32 images
-rw-r--r-- 1 root root 2.8K Jun 14 11:32 README.md
-rw-r--r-- 1 root root 6.6K Jun 14 11:32 README-TONGHOP.md
-rw-r--r-- 1 root root  303 Jun 14 11:32 requirements.txt
-rw-r--r-- 1 root root  512 Jun 14 11:32 setup.py
drwxr-xr-x 5 root root 4.0K Jun 14 11:32 tamer
-rw-r--r-- 1 root root 3.2K Jun 14 11:32 train.py
===== Config files =====
total 12K
-rw-r--r-- 1 root root 1.5K Jun 14 11:32 crohme_debug.yaml
-rw-r--r-- 1 root root 1.4K Jun 14 11:32 crohme.yaml
-rw-r--r-- 1 root root 1.3K Jun 14 11:32 hme100k.yaml
===== Data files =====
data
data/crohme
data/crohme/2019
data/crohme/2019/caption.txt
data/crohme/2019/images.pkl
data/crohme/2016
data/crohme/2016/caption.txt
data/crohme/2016/images.pkl


In [29]:
# Kiểm tra CUDA và số GPU Kaggle cấp
import sys
import torch

print("Python executable:", sys.executable)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

Python executable: /usr/bin/python3
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU count: 2
0 Tesla T4
1 Tesla T4


## 13. Chạy training

In [ ]:
# Chạy training với cấu hình CROHME
!source /kaggle/working/miniconda/bin/activate tamer && python train.py --config {CONFIG} --trainer.gpus={GPUS}

Global seed set to 7
Load data from: data/crohme
/kaggle/working/miniconda/envs/tamer/lib/python3.7/site-packages/pytorch_lightning/trainer/connectors/accelerator_connector.py:747: UserWarning: You requested multiple GPUs but did not specify a backend, e.g. `Trainer(accelerator="dp"|"ddp"|"ddp2")`. Setting `accelerator="ddp_spawn"` for you.
  "You requested multiple GPUs but did not specify a backend, e.g."
Using native 16bit precision.
GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
Global seed set to 7
initializing ddp: GLOBAL_RANK: 0, MEMBER: 1/2
Global seed set to 7
Load data from: data/crohme
Using native 16bit precision.
Global seed set to 7
initializing ddp: GLOBAL_RANK: 1, MEMBER: 2/2
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All DDP processes registered. Starting ddp with 2 processes
---------------------------------------------------

## 14. Resume training từ checkpoint nếu cần

In [ ]:
# Cell mẫu resume checkpoint từ DagsHub/Kaggle.
# !source /kaggle/working/miniconda/bin/activate tamer && python train.py --config config/crohme.yaml --trainer.resume_from_checkpoint=/kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/1-cnn-gnn/lightning_logs/version_0/checkpoints/YOUR_CHECKPOINT.ckpt --trainer.gpus=2

In [ ]:
# Cell mẫu copy checkpoint từ Kaggle input nếu cần resume.
!mkdir -p /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/1-cnn-gnn/lightning_logs/version_0/checkpoints
# !cp /kaggle/input/checkpoint/YOUR_CHECKPOINT.ckpt /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/1-cnn-gnn/lightning_logs/version_0/checkpoints/

## 15. Tìm checkpoint sau training

In [ ]:
# Liệt kê checkpoint sau training
!echo "===== Checkpoints ====="
!find lightning_logs -name "*.ckpt" -type f | sort || true

# Ghi checkpoint cuối cùng tìm được ra best_ckpt.txt để tiện eval/upload
!BEST_CKPT=$(find lightning_logs -name "*.ckpt" -type f | sort | tail -n 1); echo "$BEST_CKPT" | tee best_ckpt.txt
!echo "BEST_CKPT file content:" && cat best_ckpt.txt

## 16. Eval sau training

In [ ]:
# Eval bằng script shell có sẵn trong repo.
# Nếu eval_crohme.sh đã được cấu hình đúng trong project, cell này sẽ chạy trực tiếp.
!source /kaggle/working/miniconda/bin/activate tamer && bash eval/eval_crohme.sh

In [ ]:
# Phương án eval trực tiếp bằng eval/test.py nếu cần truyền checkpoint thủ công.
# Mặc định để comment vì cú pháp tham số phụ thuộc test.py của repo.
# BEST_CKPT=$(cat best_ckpt.txt)
# !source /kaggle/working/miniconda/bin/activate tamer && python eval/test.py --config config/crohme.yaml --checkpoint "$BEST_CKPT"

## 17. Đánh giá & Hoàn tất Đồng bộ DagsHub Cloud

Do đã sử dụng MLflow Cloud, các tệp checkpoint và logs được tự động gửi thẳng lên DagsHub MLflow Server trong suốt quá trình train. Dưới đây chỉ là hiển thị thông báo kết thúc.

In [ ]:
print("Training complete! Checkpoints are saved on your DagsHub repository Cloud Storage.")

In [ ]:
print("Check your DagsHub repository experiments tab to view metrics and download checkpoints.")

## 18. Lưu kết quả và môi trường giống template

In [ ]:
# Lưu kết quả training/eval/config để dùng cho phiên sau
!mkdir -p /kaggle/output/cnn_gnn_results
!cp -r lightning_logs /kaggle/output/cnn_gnn_results/ || true
!cp -r config /kaggle/output/cnn_gnn_results/ || true
!cp best_ckpt.txt /kaggle/output/cnn_gnn_results/ || true
!cp -r eval /kaggle/output/cnn_gnn_results/eval_files_snapshot || true
!find /kaggle/output/cnn_gnn_results -maxdepth 3 | head -100

## 19. Nén kết quả ra /kaggle/working

In [ ]:
# Nén thư mục kết quả thành file tar.gz
!tar -czvf cnn_gnn_results.tar.gz /kaggle/output/cnn_gnn_results

# Chép file nén vào thư mục working
!cp cnn_gnn_results.tar.gz /kaggle/working/

# Xác nhận file đã được chép thành công
!ls -lh /kaggle/working/cnn_gnn_results.tar.gz